# VIGIA-ML-02 — Exploracion de Dataset y Analisis Experimental de Color

**Proposito:** Notebook reproducible y desacoplado de rutas locales para:
1. Cargar `data/processed/manifest_v1.csv` y auditar dominios, sesiones y clústeres de duplicados.
2. Visualizar diferencias entre espacios `RGB/BGR`, `HSV` y `CIELAB`.
3. Probar `Baseline A` (`predict_color_classic`) sobre muestras copiadas en `data/raw/`.
4. Verificar la particion sin fugas (`GroupKFold`) y las limitaciones del sandbox.

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd

# Resolucion portable sin rutas absolutas dependientes de computadora
cwd = Path.cwd().resolve()
SANDBOX_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if not (SANDBOX_ROOT / "src").is_dir():
    raise RuntimeError(f"No se encontro src/ respecto a {cwd}. Ejecute desde experiments/vigia_ml o notebooks/.")

if str(SANDBOX_ROOT) not in sys.path:
    sys.path.insert(0, str(SANDBOX_ROOT))

print("SANDBOX_ROOT:", SANDBOX_ROOT.name)

In [ ]:
from src.color_analysis.baseline_classic import predict_color_classic
from src.models.color_features import extract_color_feature_vector

df_all = pd.read_csv(SANDBOX_ROOT / "data" / "processed" / "manifest_v1.csv")
df_raw = pd.read_csv(SANDBOX_ROOT / "data" / "processed" / "sandbox_raw_manifest_v1.csv")

print(f"Inventario global: {len(df_all)} registros | Copias deduplicadas en data/raw: {len(df_raw)}")
display(df_raw.groupby(["date_folder", "camera_id", "source_domain"])[["sample_id", "session_id", "duplicate_group"]].nunique())

In [ ]:
# Distribucion de nitidez (Laplaciana) vs luminancia (Y) por camara en copias locales
fig, ax = plt.subplots(figsize=(7, 4))
for cam, grp in df_raw.groupby("camera_id"):
    ax.scatter(grp["luminance_mean_y"], grp["sharpness_laplacian"], label=cam, alpha=0.8)
ax.set_xlabel("Luminancia media Y (0-255)")
ax.set_ylabel("Varianza Laplaciana (nitidez)")
ax.set_title("Calidad visual en copias curadas data/raw/ (NO datos reales 1:1)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Demostracion de espacios RGB vs HSV vs CIELAB y salida Baseline A sobre una muestra
sample_row = df_raw.iloc[-1]
img_bgr = cv2.imread(str(SANDBOX_ROOT / sample_row["image_path"]))
h, w = img_bgr.shape[:2]
roi = (int(0.20 * w), int(0.25 * h), int(0.80 * w), int(0.85 * h))
pred = predict_color_classic(img_bgr, roi_xyxy=roi)
feat_vec = extract_color_feature_vector(img_bgr, roi_xyxy=roi)

print("Muestra:", sample_row["sample_id"], "| Dominio:", sample_row["source_domain"])
print("Prediccion Baseline A (sin bbox anotado):", pred.predicted_color, "| Confianza:", pred.confidence)
print("Indicadores de calidad:", pred.quality_indicator)
print("Dimension del vector Baseline B:", feat_vec.shape)